In [1]:
import logging
from pathlib import Path
import sys

from modules import plotting
from modules import SequenceRepresentation as sr
from modules import training

2025-03-17 18:04:50.111665: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-17 18:04:50.221358: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-17 18:04:50.607586: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-17 18:04:50.960871: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742234691.129840   22097 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742234691.18

In [10]:
logging.basicConfig(format="%(asctime)s %(levelname)s: %(message)s", 
                    encoding='utf-8', level=logging.DEBUG)
#logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

In [3]:
wd = Path("/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative")
#wd = Path("/home/matthis/PhD/mnt/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative")
experiment_dirs = [f for f in wd.iterdir() if f.is_dir() and not (f.name.startswith("test") or f.name.startswith("slurm"))]
print(experiment_dirs[:max(3, len(experiment_dirs))])

[PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562SrfV0416101UniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562Gata2UcdUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Tead4sc101184V0422111UniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562Elk112771IggrabUniPk.narrowPeak'), PosixPath('/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEnco

In [4]:
def evaluate(experiment_dirs, evaluator_path, neg_evaluator_path = None):
    n_test_seqs = 0 # track total number of test seqs over all experiments to compare between runs, even if single experiments fail
    n_peaks = {}
    n_skipped_seqs = 0
    n_skipped_peaks = {}
    skipped_experiments = []
    glob_seqs = {}
    glob_seqs_neg = {}
    for ed in experiment_dirs:
        assert (ed / 'test_sequences_0.json').exists()
        skipping = (not (ed / evaluator_path).exists()) or (neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists())
        testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
        n_test_seqs += sum([len(g) for g in testdata])
        if skipping:
            n_skipped_seqs += sum([len(g) for g in testdata])
            skipped_experiments.append(str(ed))

        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in n_peaks:
                        n_peaks[peaksrc] = 0
                    n_peaks[peaksrc] += 1
                    if skipping:
                        if peaksrc not in n_skipped_peaks:
                            n_skipped_peaks[peaksrc] = 0
                        n_skipped_peaks[peaksrc] += 1

        if not (ed / evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / evaluator_path} does not exist")
            continue

        if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
            print(f"[WARNING] >>> skipping {ed} as {ed / neg_evaluator_path} does not exist")
            continue
        
        # retrieve peaks from test data (peaks are stored as genomic elements in the sequences)
        peaks = {}
        for g in testdata:
            for s in g:
                assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
                for e in s.genomic_elements:
                    peaksrc = e.source
                    if peaksrc not in peaks:
                        peaks[peaksrc] = {}
                    if s.id not in peaks[peaksrc]:
                        peaks[peaksrc][s.id] = []
                    peakstart, peakend = e.getRelativePositions(s)
                    assert peakend == peakstart + 1, f"Peak {e} is not a single base pair"
                    peaks[peaksrc][s.id].append(peakstart)

        # store how many times and where each sequence was hit
        seqdict = {s.id: [] for g in testdata for s in g} 
        evaluator = training.loadMultiTrainingEvaluation(str(ed / evaluator_path), testdata)
        assert len(evaluator.trainings) == 1
        tr = evaluator.trainings[0]
        for link in tr.links:
            for occs in link.occs: # list of list of occurrences
                for occ in occs:
                    assert occ.sequence.id in seqdict
                    seqdict[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        if neg_evaluator_path is not None:
            # store how many times and where each sequence was hit
            assert (ed / 'negative_test_sequences_0.json').exists()
            testdata_neg = sr.loadJSONGenomeList(str(ed / 'negative_test_sequences_0.json'))
            seqdict_neg = {s.id: [] for g in testdata_neg for s in g} 
            evaluator_neg = training.loadMultiTrainingEvaluation(str(ed / neg_evaluator_path), testdata_neg)
            assert len(evaluator_neg.trainings) == 1
            tr = evaluator_neg.trainings[0]
            for link in tr.links:
                for occs in link.occs: # list of list of occurrences
                    for occ in occs:
                        assert occ.sequence.id in seqdict_neg
                        seqdict_neg[occ.sequence.id].append((occ.position, occ.position + occ.sitelen))

        # globally count how many times each sequence was hit
        for sid in seqdict:
            if sid not in glob_seqs:
                glob_seqs[sid] = {'hits': 0, 'peaks': {}}
            glob_seqs[sid]['hits'] += len(seqdict[sid])
            for peaksrc in peaks:
                if sid in peaks[peaksrc]:
                    if peaksrc not in glob_seqs[sid]['peaks']:
                        glob_seqs[sid]['peaks'][peaksrc] = {'peaks': set(), 'hits': set()}
                    glob_seqs[sid]['peaks'][peaksrc]['peaks'].update(peaks[peaksrc][sid])
                    for p in peaks[peaksrc][sid]:
                        for hit in seqdict[sid]:
                            if hit[0] <= p < hit[1]:
                                glob_seqs[sid]['peaks'][peaksrc]['hits'].add(p)

        if neg_evaluator_path is not None:
            for sid in seqdict_neg:
                if sid not in glob_seqs_neg:
                    glob_seqs_neg[sid] = {'hits': 0}
                glob_seqs_neg[sid]['hits'] += len(seqdict_neg[sid])

    nseqs = len(glob_seqs.keys())
    nseqs_hit = len([k for k in glob_seqs.keys() if glob_seqs[k]['hits'] > 0])
    nmatches = sum([v['hits'] for v in glob_seqs.values()])

    print(f"Total number of test sequences: {n_test_seqs} | Number of peaks in these sequences: {n_peaks}")
    print(f"Skipped number of test sequences: {n_skipped_seqs} | Number of peaks in these sequences: {n_skipped_peaks}")
    print(f"Skipped {len(skipped_experiments)}/{len(experiment_dirs)} experiments: {skipped_experiments}")
    print(f"Number of sequences: {nseqs}")
    print(f"Number of sequences with hits: {nseqs_hit} | ratio: {nseqs_hit/nseqs:.2f}")
    print(f"Number of matches: {nmatches} | ratio: {nmatches/nseqs:.2f}")
    print()

    for peaksrc in peaks:
        print(f"Peak source: {peaksrc}")
        n_peak_seqs = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks']])
        n_peak_seqs_with_hits = len([k for k in glob_seqs.keys() if peaksrc in glob_seqs[k]['peaks'] and glob_seqs[k]['peaks'][peaksrc]['hits']])
        n_peaks = sum([len(v) for v in peaks[peaksrc].values()])
        n_peaks_hit = sum([len(v['peaks'][peaksrc]['hits']) for v in glob_seqs.values() if peaksrc in v['peaks']])

        print(f"Number of sequences with peaks: {n_peak_seqs}")
        print(f"Number of sequences with hits on peaks: {n_peak_seqs_with_hits} | ratio: {n_peak_seqs_with_hits/n_peak_seqs:.2f}")
        print(f"Number of peaks: {n_peaks}")
        print(f"Number of hits on peaks: {n_peaks_hit} | ratio: {n_peaks_hit/n_peaks:.2f}")
        print()

    if neg_evaluator_path is not None:
        nseqs_neg = len(glob_seqs_neg.keys())
        nseqs_hit_neg = len([k for k in glob_seqs_neg.keys() if glob_seqs_neg[k]['hits'] > 0])
        nmatches_neg = sum([v['hits'] for v in glob_seqs_neg.values()])

        print(f"Number of negative sequences: {nseqs_neg}")
        print(f"Number of negative sequences with hits: {nseqs_hit_neg} | ratio: {nseqs_hit_neg/nseqs_neg:.2f}")
        print(f"Number of negative matches: {nmatches_neg} | ratio: {nmatches_neg/nseqs_neg:.2f}")
        print()

In [5]:
evaluate(experiment_dirs, 'evaluator_test.json', 'evaluator_negative_test.json')

2025-03-17 18:05:05,913 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)


2025-03-17 18:05:06,360 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)


2025-03-17 18:05:08,980 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:42295995-42296645:0:0-650 (2x), chr2:145089806-145090327:0:0-521 (2x), chr3:42641896-42642557:0:0-661 (2x)


...

[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr4:128702821-128703160:0:0-339 (2x), negative_chr7:100136668-100137233:0:0-565 (2x), negative_chr22:19705322-19706321:0:0-999 (2x), negative_chr1:197871671-197872361:0:0-690 (2x), negative_chr1:43123582-43124140:0:0-558 (2x), negative_chr17:61926089-61927170:0:0-1,081 (2x), negative_chr2:106015376-106015966:0:0-590 (2x), negative_chr22:20067214-20068089:0:0-875 (2x), negative_chr9:37485657-37486115:0:0-458 (2x), negative_chr1:155022843-155023312:0:0-469 (2x), negative_chr11:118868258-118868750:0:0-492 (2x), negative_chr7:135346967-135347443:0:0-476 (2x), negative_chr11:67764043-67764415:0:0-372 (2x), negative_chr7:148725666-148726250:0:0-584 (2x), negative_chr19:12917200-12917697:0:0-497 (2x), negative_chr16:89939464-89940035:0:0-571 (2x), negative_chr13:115079619-115080065:0:0-446 (2x), negative_chr16:81129904-81130331:0:0-427 (2x), negative_chr11:125495140-125495604:0:0-464 (2x), negative_chr1:12

2025-03-17 18:05:41,271 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr19:39340544-39341230:0:0-686 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr19:39340544-39341230:0:0-686 (2x)


2025-03-17 18:05:41,308 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr19:39340544-39341230:0:0-686 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr19:39340544-39341230:0:0-686 (2x)
Total number of test sequences: 216125 | Number of peaks in these sequences: {'bed.tsv': 217928, 'fimo.tsv': 85725, 'mast.tsv': 93968}
Skipped number of test sequences: 0 | Number of peaks in these sequences: {}
Skipped 0/40 experiments: []
Number of sequences: 215834
Number of sequences with hits: 121702 | ratio: 0.56
Number of matches: 284647 | ratio: 1.32

Peak source: bed.tsv
Number of sequences with peaks: 215808
Number of sequences with hits on peaks: 20487 | ratio: 0.09
Number of peaks: 941
Number of hits on peaks: 20494 | ratio: 21.78

Peak source: fimo.tsv
Number of sequences with peaks: 84777
Number of sequences with hits on peaks: 28794 | ratio: 0.34
Number of peaks: 488
Number of hits on peaks: 28794 | ratio: 59.00

Peak source: mast.tsv
Number of sequences with peaks: 81851
Number of sequences with hits on peaks: 29450 | ratio: 0.36
Number of peaks: 7

In [6]:
evaluate(experiment_dirs, 'STREME/streme_evaluator_dummymodel_test.json', 'STREME/streme_evaluator_dummymodel_negative_test.json')

2025-03-17 18:05:43,398 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr2:46543708-46544075:0:0-367 (2x), chr10:11220561-11220973:0:0-412 (2x), chr17:56736279-56736881:0:0-602 (2x)


2025-03-17 18:05:43,803 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr2:46543708-46544075:0:0-367 (2x), negative_chr10:11220561-11220973:0:0-412 (2x), negative_chr17:56736279-56736881:0:0-602 (2x)
[WARNING] >>> skipping /home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak as /home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
[WARNING] >>> skipping /home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak as /home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak/STREME/streme_evaluator_dummymodel_test.json does not exist
[WARNING] >>> skipping /home/ebelm/brain/genomegraph/runs/20250306_new_expe

2025-03-17 18:05:47,834 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr17:72510161-72511226:0:0-1,065 (2x), chr6:27655855-27656374:0:0-519 (2x)


...

[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr4:128702821-128703160:0:0-339 (2x), negative_chr7:100136668-100137233:0:0-565 (2x), negative_chr22:19705322-19706321:0:0-999 (2x), negative_chr1:197871671-197872361:0:0-690 (2x), negative_chr1:43123582-43124140:0:0-558 (2x), negative_chr17:61926089-61927170:0:0-1,081 (2x), negative_chr2:106015376-106015966:0:0-590 (2x), negative_chr22:20067214-20068089:0:0-875 (2x), negative_chr9:37485657-37486115:0:0-458 (2x), negative_chr1:155022843-155023312:0:0-469 (2x), negative_chr11:118868258-118868750:0:0-492 (2x), negative_chr7:135346967-135347443:0:0-476 (2x), negative_chr11:67764043-67764415:0:0-372 (2x), negative_chr7:148725666-148726250:0:0-584 (2x), negative_chr19:12917200-12917697:0:0-497 (2x), negative_chr16:89939464-89940035:0:0-571 (2x), negative_chr13:115079619-115080065:0:0-446 (2x), negative_chr16:81129904-81130331:0:0-427 (2x), negative_chr11:125495140-125495604:0:0-464 (2x), negative_chr1:12

2025-03-17 18:06:08,327 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr19:39340544-39341230:0:0-686 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: chr19:39340544-39341230:0:0-686 (2x)


2025-03-17 18:06:08,353 WARNING: [loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr19:39340544-39341230:0:0-686 (2x)


[loadMultiTrainingEvaluation] >>> Duplicate sequence IDs found in allGenomes: negative_chr19:39340544-39341230:0:0-686 (2x)
Total number of test sequences: 216125 | Number of peaks in these sequences: {'bed.tsv': 217928, 'fimo.tsv': 85725, 'mast.tsv': 93968}
Skipped number of test sequences: 83963 | Number of peaks in these sequences: {'bed.tsv': 84108, 'fimo.tsv': 51512, 'mast.tsv': 55107}
Skipped 7/40 experiments: ['/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsSydhK562MaffIggrabUniPk.narrowPeak', '/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Egr1V0416101UniPk.narrowPeak', '/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsUtaK562CtcfUniPk.narrowPeak', '/home/ebelm/brain/genomegraph/runs/20250306_new_experiments/20250313_test_negative/wgEncodeAwgTfbsHaibK562Elf1sc631V0416102UniPk.narrowPeak', '/home/ebelm/brain/genomegraph/ru

---

Distribution of test sequence number and peaks per experiment, seems to be quite unevenly distributed.

In [7]:
[0]*2

[0, 0]

In [8]:
n_test_seqs = []
n_peaks = {}
i_skipped = set()

evaluator_path = "STREME/streme_evaluator_dummymodel_test.json"
neg_evaluator_path = "STREME/streme_evaluator_dummymodel_negative_test.json"

for i, ed in enumerate(experiment_dirs):
    assert (ed / 'test_sequences_0.json').exists()
    testdata = sr.loadJSONGenomeList(str(ed / 'test_sequences_0.json'))
    skipping = (not (ed / evaluator_path).exists()) or (neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists())

    n_test_seqs.append( sum([len(g) for g in testdata]) )

    for g in testdata:
        for s in g:
            assert s.elementsPossible(), f"Sequence {s.id} can't contain elements"
            for e in s.genomic_elements:
                peaksrc = e.source
                if peaksrc not in n_peaks:
                    n_peaks[peaksrc] = [0]*i # in the (unlikely) case that experiments [0, i) did not have this type
                assert len(n_peaks[peaksrc]) >= i, f"{peaksrc}, {i}, {n_peaks}" # should not fail
                if len(n_peaks[peaksrc]) == i:
                    n_peaks[peaksrc].append(0)

                n_peaks[peaksrc][i] += 1

    if not (ed / evaluator_path).exists():
        i_skipped.add(i)
        continue

    if neg_evaluator_path is not None and not (ed / neg_evaluator_path).exists():
        i_skipped.add(i)
        continue

In [9]:
plotting.ownPlotlyHist(lists={'test seqs': n_test_seqs})#, bins=list(range(0,max(n_test_seqs)+1)))

2025-03-17 18:06:27,945 DEBUG: [plotting.ownHist_impl] called with ls=[390, 1416, 5004, 8334, 8721, 3218, 661, 2455, 792, 3097, 1690, 13852, 925, 4894, 5797, 938, 12975, 16818, 7218, 5557, 4804, 1223, 2506, 7371, 1485, 9431, 3804, 11100, 9309, 6750, 11615, 7523, 4460, 15598, 1264, 370, 2162, 6514, 889, 3195], binSize=None, bins=None


[plotting.ownHist_impl] called with ls=[390, 1416, 5004, 8334, 8721, 3218, 661, 2455, 792, 3097, 1690, 13852, 925, 4894, 5797, 938, 12975, 16818, 7218, 5557, 4804, 1223, 2506, 7371, 1485, 9431, 3804, 11100, 9309, 6750, 11615, 7523, 4460, 15598, 1264, 370, 2162, 6514, 889, 3195], binSize=None, bins=None


2025-03-17 18:06:27,948 DEBUG: [plotting.ownHist_impl] called with ls=[1416, 3195, 9309, 7523, 11100, 889, 16818, 4804, 925, 3218, 6750, 792, 1223, 5797, 370, 8334, 5557, 7371, 3804, 13852, 7218, 390, 661, 15598, 1690, 2162, 2506, 5004, 11615, 3097, 6514, 1485, 4460, 2455, 9431, 8721, 12975, 1264, 4894, 938], binSize=None, bins=[328.96, 493.43999999999994, 657.92, 822.4, 986.8799999999999, 1151.36, 1315.84, 1480.32, 1644.8, 1809.28, 1973.7599999999998, 2138.24, 2302.72, 2467.2, 2631.68, 2796.16, 2960.64, 3125.12, 3289.6, 3454.08, 3618.56, 3783.04, 3947.5199999999995, 4112.0, 4276.48, 4440.96, 4605.44, 4769.92, 4934.4, 5098.88, 5263.36, 5427.839999999999, 5592.32, 5756.799999999999, 5921.28, 6085.759999999999, 6250.24, 6414.719999999999, 6579.2, 6743.679999999999, 6908.16, 7072.639999999999, 7237.12, 7401.599999999999, 7566.08, 7730.5599999999995, 7895.039999999999, 8059.5199999999995, 8224.0, 8388.48, 8552.96, 8717.439999999999, 8881.92, 9046.4, 9210.88, 9375.359999999999, 9539.84, 970

[plotting.ownHist_impl] called with ls=[1416, 3195, 9309, 7523, 11100, 889, 16818, 4804, 925, 3218, 6750, 792, 1223, 5797, 370, 8334, 5557, 7371, 3804, 13852, 7218, 390, 661, 15598, 1690, 2162, 2506, 5004, 11615, 3097, 6514, 1485, 4460, 2455, 9431, 8721, 12975, 1264, 4894, 938], binSize=None, bins=[328.96, 493.43999999999994, 657.92, 822.4, 986.8799999999999, 1151.36, 1315.84, 1480.32, 1644.8, 1809.28, 1973.7599999999998, 2138.24, 2302.72, 2467.2, 2631.68, 2796.16, 2960.64, 3125.12, 3289.6, 3454.08, 3618.56, 3783.04, 3947.5199999999995, 4112.0, 4276.48, 4440.96, 4605.44, 4769.92, 4934.4, 5098.88, 5263.36, 5427.839999999999, 5592.32, 5756.799999999999, 5921.28, 6085.759999999999, 6250.24, 6414.719999999999, 6579.2, 6743.679999999999, 6908.16, 7072.639999999999, 7237.12, 7401.599999999999, 7566.08, 7730.5599999999995, 7895.039999999999, 8059.5199999999995, 8224.0, 8388.48, 8552.96, 8717.439999999999, 8881.92, 9046.4, 9210.88, 9375.359999999999, 9539.84, 9704.32, 9868.8, 10033.27999999999

AssertionError: first bin must be convertible to int, but is 328.96